# CAM — Simulations for Section 7 of the paper

Reproducibility notebook for *"Conformal Adaptive Martingales"* — one cell per subsection of Section 7 (Simulation study).

**Requirements**: `cam_grid6.py` (final version, deterministic seeds via `zlib.crc32`) (included in this repository). All runs use fixed seeds: re-executing this notebook with the same script reproduces exactly the tables and figures in the paper.

**Runtime warning**: cells 7.3 and 7.4 take hours on standard Colab. Outputs (CSV + PNG) are written to `cam_output/` as soon as each experiment completes, so a disconnection does not lose finished experiments. Execute the cells in order.

## 0. Setup

Locate the repository (local run or Colab clone), import the library, set the output folder, and verify the seed signature.

In [ ]:
import importlib, os, sys
from IPython.display import Image, display

# Works both locally (repository root) and on Google Colab.
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Option A: clone the repository into the Colab session
    if not os.path.exists('REPO'):
        !git clone https://github.com/<USER>/<REPO>.git REPO
    BASE = os.path.abspath('REPO')
else:
    BASE = os.path.abspath('.')
sys.path.insert(0, BASE)

import cam_grid6 as m
importlib.reload(m)

DIR = os.path.join(BASE, 'cam_output')
os.makedirs(DIR, exist_ok=True)
m.OUTPUT_DIR = DIR             # every save_df/save_figure call writes here
print('output ->', DIR)

In [ ]:
# Reproducibility check: the crc32 seed signature must be 298, 553, 219, 604
# on any machine and in any session. If it differs, the script is not the final version.
import zlib
sig = [zlib.crc32(d.encode()) % 1000 for d in ('Normal', 'Gamma(k=2)', 'Exponential', 'Gumbel(max)')]
print(sig)
assert sig == [298, 553, 219, 604], 'script mismatch: use the final cam_grid6'

## 7.1 Design — mechanism figure (`fig_pvalue_hist`)

Histograms of the online conformal p-values, in control (uniform) and after the shift: the object every conformal detector operates on. Paper: Figure `fig:pvalue_hist`.

In [ ]:
p_hist = m.make_pvalue_hist(dist='Exponential', delta=1.0, save_dir=DIR)
display(Image(p_hist))

## 7.2 In-control — anytime false alarm probability (`fig_sim1_fa`)

Empirical curves $P_0(\tau \le m)$ against the theoretical bounds: $2m/\gamma$ for the two-sided CAM (Theorem), $m/\gamma$ for the one-sided power betting. Standard normal in-control distribution; by Lemma 2.9 the run-length law is the same for every continuous $F_0$. Paper: Figure `fig:sim1_fa`. Runtime: tens of minutes.

In [ ]:
p_fa = m.make_sim1_paper_figure(gamma=200, horizon=600, reps=4000)
print(p_fa)
display(Image(p_fa['png']))

## 7.2 In-control — nominal versus realised ARL (`fig_sit1_arl`)

Thresholds calibrated once on the normal and applied to every distribution: the conformal detector keeps ARL = 200 everywhere, the Gaussian CUSUM over-alarms on the skewed distributions. Paper: Figure `fig:sit1_arl` + `table_sit1_arl_nominal_vs_realized.csv`.

In [ ]:
res1 = m.sim1_false_alarm_grid(gamma=200, horizon=2000, reps=4000)
display(res1['table'])
p_sit1 = m.make_sit1_figure(res1, style='bars', save_dir=DIR)
display(Image(p_sit1))

## 7.3 Out-of-control — delay at matched ARL (`tab:sim2_delay`)

Gaussian mean shift, $\nu = 1$, six methods (two-sided CAM = reference at $\gamma = 200$; all others calibrated by bisection to its realised ARL). Produces `table_sim2_delay_matched_arl.csv` and `table_sim2_calibration.csv`. Runtime: hours (calibration dominates).

In [ ]:
res2 = m.sim2_delay_matched_arl(
    gamma_design=200,
    horizon_cal=2500, reps_cal=1500,   # Steps A+B: target ARL + threshold calibration
    horizon_del=2000, reps_del=4000,   # Step C: delay measurement
)
display(res2['calibration'])   # thresholds and realised ARL -> table_sim2_calibration.csv
display(res2['table'])         # delays per method and delta -> table_sim2_delay_matched_arl.csv

## 7.4 Industrial grid (`fig_regime`, `tab:grid_cells`)

12 cells = 4 in-control distributions × 3 shift magnitudes, changepoint at $\nu = 100$, matched ARL per directional family (two-sided on the Normal, one-sided on the skewed distributions). Produces `table_grid_level_shift.csv` and `table_grid_summary.csv`. Runtime: the longest cell of the notebook (hours).

In [ ]:
res3 = m.sim_level_shift_grid(
    dists=('Normal', 'Gamma(k=2)', 'Exponential', 'Gumbel(max)'),
    deltas=(0.5, 1.0, 2.0), nu=100,
    horizon_cal=2500, reps_cal=1500,
    horizon_del=2200, reps_del=4000,
    save=True, verbose=True,
)
print('common (target) ARL:', round(res3['target_arl'], 1))
display(res3['summary'])
display(res3['table'])

## 7.4 Grid figures (`fig_regime`, `fig_nu_dilution`)

`fig_regime` is derived from `res3` with no recomputation; the $\nu$-sweep (Gamma, $\delta = 1$, $\nu \in \{1, 25, 50, 100, 200, 400\}$) re-simulates with the calibration aligned to the grid (2500/1500). Run in the same session as the previous cell.

In [ ]:
paths = m.make_figures(
    res3, save_dir=DIR,
    horizon_cal=2500, reps_cal=1500,   # align the sweep calibration with the grid
    reps_sweep=1000,
)
print(paths)
display(Image(paths['regime']))
display(Image(paths['nu_dilution']))

## 7.5 Phase II ablation (`tab:sim3_phase2`)

Three Phase II estimators (histogram, Beta-MoM, Chen-KDE) at nominal thresholds $\gamma \in \{100, 200, 500\}$, small shift $\delta = 0.5$: validity does not depend on the estimator, so the comparison measures delay only. Produces `table_sim3_phase2_estimators.csv`.

In [ ]:
res_s3 = m.sim3_phase2_estimators(gammas=(100, 200, 500), delta=0.5,
                                 horizon=2500, reps=4000)
display(res_s3['table'])

## 7.5b Matched-ARL check ("A direct check confirms this reading")

Reproduces the check cited in the text of 7.5: recalibrates the histogram variants to the realised ARL of the Beta default (same seeds and paths as sim2, paired comparison) and evaluates them on the $\delta = 0.5$ cell. Expected: hist thresholds $\approx 261/271$, delays $174.3$ vs $171.5$ (two-sided) and $138.3$ vs $136.3$ (one-sided) — equivalence at matched risk. Produces `table_sim3_matched_arl_check.csv`.

In [ ]:
res_check = m.sim3_matched_arl_check(
    gamma_design=200, delta=0.5,
    horizon_cal=2500, reps_cal=1500,
    horizon_del=2000, reps_del=4000,
)
print('target ARL:', round(res_check['target_arl'], 1))
display(res_check['table'])

## 7.6 Cost of two-sidedness (`tab:sim4`)

CAM-smallP, CAM-largeP and CAM-twoSided at matched ARL on Gaussian shifts of both signs ($\delta = \pm 1$, $\nu = 1$). Expected pattern: correctly oriented component fast, mis-oriented component in gating failure, two-sided within $\approx \log 2 / I$ of the best (consistency with the decomposition of 7.3: margin $\approx 2.5$ observations at $\delta = 1$). Produces `table_sim4_one_sided_vs_two_sided.csv`.

In [ ]:
res4 = m.sim4_one_sided_vs_two_sided(horizon_cal=2500, reps_cal=1500,
                                     horizon_del=1000, reps_del=4000)
display(res4['table'])

## Final archive

List of the outputs on Drive and a downloadable zip with all figures and tables.

## Regenerate the figure files without re-running the long experiments

If the CSV tables of a previous run are already in `cam_output/`, this cell redraws Figures 1–4 and S1 as PNG and vector PDF. Figures 2 and 3 are redrawn from the saved tables; Figure 4 re-runs only the changepoint sweep (~1 hour); Figure 1 re-runs the in-control paths (tens of minutes).

In [ ]:
import pandas as pd

t_sit1 = pd.read_csv(os.path.join(DIR, 'table_sit1_arl_nominal_vs_realized.csv'))
t_grid = pd.read_csv(os.path.join(DIR, 'table_grid_level_shift.csv'))

print(m.make_sim1_paper_figure(gamma=200, horizon=600, reps=4000))            # Figure 1
print(m.make_sit1_figure({'table': t_sit1}, style='bars', save_dir=DIR))       # Figure 2
print(m.make_figures({'table': t_grid}, save_dir=DIR,                          # Figures 3-4
                     horizon_cal=2500, reps_cal=1500, reps_sweep=1000))
print(m.make_pvalue_hist(dist='Exponential', delta=1.0, save_dir=DIR))         # Figure S1

In [ ]:
import glob, shutil
for f in sorted(glob.glob(os.path.join(DIR, '*'))):
    print(os.path.basename(f))

shutil.make_archive(os.path.join(BASE, 'cam_output'), 'zip', DIR)
if IN_COLAB:
    from google.colab import files
    files.download(os.path.join(BASE, 'cam_output.zip'))